# 05 DCF, SOTP, Sensitivities and Memo

Final question: at Tesla's current market valuation, what has to go right?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/tesla_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/tesla_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
options = pd.read_csv(DATA_DIR / "option_scenarios.csv")
premiums = pd.read_csv(DATA_DIR / "market_premiums.csv").set_index("case")
cases = ["bear", "base", "bull"]

In [ ]:
def case_table(segment):
    return (
        assumptions[assumptions["segment"].eq(segment)]
        .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
        .reindex(cases)
    )


auto = case_table("Auto")
auto["revenue_usd_bn"] = auto["2030 deliveries"] * auto["ASP"] / 1_000
auto["ebitda_usd_bn"] = auto["revenue_usd_bn"] * auto["EBITDA margin"]
auto["selected_value_usd_bn"] = auto["revenue_usd_bn"] * [1.5, 2.5, 4.0]

energy = case_table("Energy")
energy["ebitda_usd_bn"] = energy["2030 revenue"] * energy["EBITDA margin"]
energy["selected_value_usd_bn"] = energy["2030 revenue"] * [2.0, 4.0, 6.0]

services = case_table("Services")
services["ebitda_usd_bn"] = services["2030 revenue"] * services["EBITDA margin"]
services["selected_value_usd_bn"] = services["2030 revenue"] * [1.5, 2.5, 4.0]

fsd = case_table("FSD")
fsd["ebitda_usd_bn"] = fsd["2030 revenue"] * fsd["EBITDA margin"]
fsd["selected_value_usd_bn"] = fsd["2030 revenue"] * [5.0, 10.0, 15.0]

option_values = {}
for option_name, group in options.groupby("option"):
    scenarios = [
        OptionScenario(row.case, row.probability, row.value_usd_bn)
        for row in group.itertuples(index=False)
    ]
    option_values[option_name] = probability_weighted_option_value(scenarios)
option_values

In [ ]:
capital = assumptions[assumptions["segment"].eq("Capital")].pivot_table(
    index="case", columns="metric", values="value", aggfunc="first"
)
sotp_rows = []
for case in cases:
    segments = [
        SegmentValuation("Auto", auto.loc[case, "selected_value_usd_bn"], "EV/Revenue"),
        SegmentValuation("Energy", energy.loc[case, "selected_value_usd_bn"], "EV/Revenue"),
        SegmentValuation(
            "Services/Supercharging", services.loc[case, "selected_value_usd_bn"], "EV/Revenue"
        ),
        SegmentValuation("FSD/software", fsd.loc[case, "selected_value_usd_bn"], "EV/Revenue"),
        SegmentValuation(
            "Robotaxi option", option_values["Robotaxi"], "Probability weighted option"
        ),
        SegmentValuation("Optimus option", option_values["Optimus"], "Probability weighted option"),
    ]
    bridge = sotp_equity_value(
        SotpInputs(
            segments=segments,
            cash=capital.loc["base", "cash"],
            debt=capital.loc["base", "debt"],
            dilution=capital.loc["base", "dilution"],
            fully_diluted_shares=capital.loc["base", "fully diluted shares"],
        )
    )
    premium = premiums.loc[case]
    market = market_premium_value(
        bridge.equity_value,
        MarketPremiumInputs(
            musk_premium=premium.musk_premium,
            ai_scarcity_premium=premium.ai_autonomy_premium,
            ipo_scarcity_premium=premium.mega_cap_liquidity_premium,
            strategic_asset_premium=premium.energy_growth_premium,
            governance_discount=premium.governance_discount,
            execution_haircut=premium.execution_haircut,
        ),
    )
    sotp_rows.append(
        {
            "case": case,
            "fundamental_sotp_usd_bn": bridge.equity_value,
            "net_premium_discount": market.net_premium,
            "market_implied_value_usd_bn": market.market_value,
            "value_per_share_usd": market.market_value
            / capital.loc["base", "fully diluted shares"],
        }
    )
sotp = pd.DataFrame(sotp_rows).set_index("case")
sotp

In [ ]:
reverse_grid = build_sensitivity_grid(
    row_values=[150, 200, 250, 300, 350, 400],
    column_values=[0.12, 0.18, 0.24, 0.30],
    formula=lambda revenue, margin: revenue * margin * 25.0,
    row_name="2030 revenue (USD bn)",
    column_name="EBITDA margin",
)
reverse_grid

In [ ]:
current_equity_value = 1_400.0
implied_requirements = pd.DataFrame(
    [
        {
            "exit_multiple": multiple,
            "target_margin": margin,
            "required_2030_revenue_usd_bn": implied_revenue_for_enterprise_value(
                current_equity_value, margin, multiple
            ),
            "required_margin_at_250bn_revenue": implied_margin_for_enterprise_value(
                current_equity_value, 250.0, multiple
            ),
        }
        for multiple in [18.0, 25.0, 32.0]
        for margin in [0.18, 0.24, 0.30]
    ]
)
implied_requirements

In [ ]:
football = pd.DataFrame(
    {
        "method": [
            "Auto cash flow",
            "Energy storage",
            "Services / charging",
            "FSD software",
            "SOTP",
            "Market premium",
            "Bull option case",
        ],
        "low": [
            80,
            70,
            35,
            40,
            sotp.loc["bear", "fundamental_sotp_usd_bn"],
            sotp.loc["bear", "market_implied_value_usd_bn"],
            900,
        ],
        "high": [
            1_720,
            840,
            300,
            1_200,
            sotp.loc["bull", "fundamental_sotp_usd_bn"],
            sotp.loc["bull", "market_implied_value_usd_bn"],
            3_000,
        ],
    }
)
ax = football.plot.barh(
    x="method", y=["low", "high"], figsize=(9, 5), title="Tesla Valuation Football Field"
)
ax.set_xlabel("USD bn")
plt.tight_layout()
football

## Research Memo Draft

At Tesla's current market valuation, the stock is not being valued as a
normal auto OEM. The valuation debate is whether auto cash flow, energy
storage scale, recurring software, autonomy and robotics can compound into
a platform-like earnings base before market patience fades.

What has to go right:

- Auto deliveries must recover without requiring structurally lower margins.
- Energy storage must become a large, profitable second cash-flow engine.
- FSD must convert from feature promise into high-margin recurring software.
- Robotaxi must prove real unit economics before it deserves base-case credit.
- Optimus must show source-backed milestones before large value moves out of option value.
- Public markets must keep awarding Tesla an AI/autonomy premium despite governance and execution risk.

This is research and modeling support, not a recommendation or portfolio action.